## TASK 2: Emotion Recognition from Speech 

In [8]:
import os
import glob
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [9]:
DATA_PATH = "audio_set"

In [10]:

def extract_features_and_labels(data_path):
    X = []
    y = []
    
    # Target path pattern to grab all .wav files across actor directories
    search_path = os.path.join(data_path, "Actor_*", "*.wav")
    file_list = glob.glob(search_path)
    
    if not file_list:
        print("No files found! Check your data_path.")
        return None, None

    print(f"Found {len(file_list)} audio files. Extracting features...")

    for file_path in file_list:
        # 1. Parse the filename to extract the emotion label
        file_name = os.path.basename(file_path)
        part = file_name.split('-')
        emotion_code = int(part[2])  # The 3rd identifier
        
        # 2. Load audio and extract MFCCs
        try:
            # kaiser_fast makes loading significantly quicker
            audio, sample_rate = librosa.load(file_path)
            
            # Extract 40 MFCC coefficients
            mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
            
            # Average across the time axis to get a consistent 1D vector of length 40
            mfccs_scaled = np.mean(mfccs.T, axis=0)
            
            X.append(mfccs_scaled)
            y.append(emotion_code)
        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            continue

    return np.array(X), np.array(y)

In [11]:
X, y = extract_features_and_labels(DATA_PATH)
print(f"Feature matrix shape: {X.shape}")  # Should be (1440, 40)
print(f"Labels array shape: {y.shape}")    # Should be (1440,)

Found 1440 audio files. Extracting features...
Feature matrix shape: (1440, 40)
Labels array shape: (1440,)


In [13]:
# 1. Shift labels down by 1 (so they map from 0 to 7)
y_adjusted = y - 1

# 2. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_adjusted, test_size=0.2, random_state=42, stratify=y_adjusted)

# 3. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Pure NumPy alternative to to_categorical (One-Hot Encoding)
#y_train_encoded = np.eye(8)[y_train]
#y_test_encoded = np.eye(8)[y_test]

print("Data preparation complete for PyTorch!")
#print(f"X_train shape: {X_train_scaled.shape}, y_train shape: {y_train_encoded.shape}")

Data preparation complete for PyTorch!


In [14]:
# Convert our NumPy arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)  # PyTorch CrossEntropy expects class indices (0-7), not one-hot encoding

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoaders for easy batching
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [15]:
# Define the Architecture
class EmotionMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(EmotionMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes) # Out nodes = 8 emotions
        )
        
    def forward(self, x):
        return self.network(x)

In [16]:
# Initialize the model, loss function, and optimizer
model = EmotionMLP(input_dim=40, num_classes=8)
criterion = nn.CrossEntropyLoss() # Handles softmax internally
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

EmotionMLP(
  (network): Sequential(
    (0): Linear(in_features=40, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Linear(in_features=64, out_features=8, bias=True)
  )
)


In [18]:
epochs = 50

for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        # 1. Clear gradients from the previous step
        optimizer.zero_grad()
        
        # 2. Forward pass: compute predicted outputs
        outputs = model(batch_X)
        
        # 3. Calculate the loss
        loss = criterion(outputs, batch_y)
        
        # 4. Backward pass: compute gradient of the loss with respect to model parameters
        loss.backward()
        
        # 5. Perform a single optimization step (parameter update)
        optimizer.step()
        
        # Track statistics
        running_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct / total) * 100
    
    # Evaluate on test data every 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        model.eval() # Set model to evaluation mode
        with torch.no_grad():
            test_outputs = model(X_test_tensor)
            _, test_predicted = torch.max(test_outputs, 1)
            test_acc = (test_predicted == y_test_tensor).sum().item() / y_test_tensor.size(0) * 100
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}% - Test Acc: {test_acc:.2f}%")

Epoch [1/50] - Loss: 0.3039 - Train Acc: 88.54% - Test Acc: 72.57%
Epoch [10/50] - Loss: 0.1939 - Train Acc: 93.58% - Test Acc: 74.31%
Epoch [20/50] - Loss: 0.1922 - Train Acc: 93.84% - Test Acc: 71.18%
Epoch [30/50] - Loss: 0.1502 - Train Acc: 94.97% - Test Acc: 72.22%
Epoch [40/50] - Loss: 0.1433 - Train Acc: 94.97% - Test Acc: 73.61%
Epoch [50/50] - Loss: 0.1037 - Train Acc: 95.83% - Test Acc: 75.35%


In [20]:
emotions_map = {
    0: "Neutral",
    1: "Calm",
    2: "Happy",
    3: "Sad",
    4: "Angry",
    5: "Fearful",
    6: "Disgust",
    7: "Surprised"
}

def predict_custom_audio(file_path, trained_model, data_scaler):
    # Set model to evaluation mode (turns off dropout)
    trained_model.eval()
    
    try:
        # 2. Load the custom audio file
        audio, sample_rate = librosa.load(file_path)
        
        # 3. Extract the 40 MFCC features (Exactly like the training data)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        mfccs_processed = np.mean(mfccs.T, axis=0)
        
        # 4. Reshape to a 2D array because the scaler expects a matrix input
        mfccs_reshaped = mfccs_processed.reshape(1, -1)
        
        # 5. Scale the features using the *same* scaler instance from training
        mfccs_scaled = data_scaler.transform(mfccs_reshaped)
        
        # 6. Convert to PyTorch Tensor
        input_tensor = torch.tensor(mfccs_scaled, dtype=torch.float32)
        
        # 7. Make the prediction without calculating gradients
        with torch.no_grad():
            outputs = trained_model(input_tensor)
            
            # Convert raw outputs (logits) into clean percentage probabilities
            probabilities = torch.softmax(outputs, dim=1)[0]
            
            # Find the index with the highest probability
            _, predicted_idx = torch.max(outputs, 1)
            predicted_label = predicted_idx.item()
            
        # 8. Print out results 
        print("--- Prediction Analysis ---")
        print(f"Predicted Emotion: **{emotions_map[predicted_label]}**\n")
        print("Confidence Breakdown:")
        for idx, prob in emotions_map.items():
            print(f" - {prob}: {probabilities[idx].item() * 100:.2f}%")
            
    except Exception as e:
        print(f"Error reading or processing custom audio file: {e}")

predict_custom_audio("Recording (9).wav", model, scaler)
predict_custom_audio("Recording (11).wav", model, scaler)

--- Prediction Analysis ---
Predicted Emotion: **Disgust**

Confidence Breakdown:
 - Neutral: 0.00%
 - Calm: 0.00%
 - Happy: 0.00%
 - Sad: 0.00%
 - Angry: 0.00%
 - Fearful: 0.00%
 - Disgust: 100.00%
 - Surprised: 0.00%
--- Prediction Analysis ---
Predicted Emotion: **Sad**

Confidence Breakdown:
 - Neutral: 0.00%
 - Calm: 0.00%
 - Happy: 0.80%
 - Sad: 98.71%
 - Angry: 0.00%
 - Fearful: 0.00%
 - Disgust: 0.00%
 - Surprised: 0.50%
